# Lesson 2 — Design of experiments & batch data generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karefyllidis/PlugFlowML/blob/main/notebooks/Main_2_generate_training_data.ipynb)

*Part of the [PlugFlowML course](../README.md): machine-learning surrogates for physical simulation, taught on a chemical reactor.*

**Mode:** hands-on (Colab-ready) · **Runtime:** ~1-2 min for the demo config shipped in this repo (10 runs, ~8-9 s each); scales roughly linearly with run count for a larger campaign · **Builds on:** Lesson 1

**What you'll learn**

1. Choose a sampling design for an expensive simulator — Latin Hypercube vs grid vs random — and justify the choice in terms of coverage per run.
2. Turn a single simulation into a batch campaign: parameter bounds, run budget, incremental saves, and metadata for reproducibility.
3. Treat failed runs as a normal, informative part of a simulation campaign, and know what to check before training on the survivors.
4. Recognise an embarrassingly parallel workload and how such campaigns are chunked across an HPC cluster (the SLURM-array pattern).

**The example system.** This notebook wraps Lesson 1's single PFR run in a loop. Six inlet/geometry parameters — feed temperature, pressure, tube length and diameter, mass flow rate, and wall heat flux — are sampled within configured bounds; each sample becomes one Cantera simulation, and each simulation contributes ~200 axial rows of 245+ features, saved incrementally with a metadata record for reproducibility. The dataset used in the rest of the course came from exactly this notebook run at scale: 50,000 n-hexane simulations on an HPC cluster (~92% of which succeeded), yielding ~9.2 million rows.

In [ ]:
# ══ 0. COLAB BOOTSTRAP ══
# Running locally? This cell is a no-op — just run it and move on.
import sys, subprocess
from pathlib import Path

if "google.colab" in sys.modules:
    if Path.cwd().name != "notebooks":            # fresh Colab VM
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/karefyllidis/PlugFlowML.git"], check=True)
        %cd PlugFlowML/notebooks
    %pip install -q cantera

In [ ]:
# Setup and Imports
import sys
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from datetime import datetime

# Project root: notebooks live under notebooks/; from there, project root is one level up
current_dir = Path(os.getcwd())
if (current_dir / 'src').exists():
    project_root = current_dir
elif (current_dir.parent / 'src').exists():
    project_root = current_dir.parent
else:
    project_root = current_dir
os.chdir(project_root)  # so configs/, data/training/, mechanisms/ resolve correctly

# IMPORTANT: Import cantera BEFORE adding src to sys.path
# This prevents namespace conflict: without this, Python would find src/cantera/
# (our package) instead of the actual cantera library
import cantera as ct
print(f"Cantera version: {ct.__version__}")

# Suppress all Cantera/SUNDIALS warnings and messages
import warnings
import logging

# Add project root to path (after cantera is imported)
sys.path.insert(0, str(project_root))

from src.ml.data_generation import TrainingDataGenerator

print("PlugFlowML: ML Training Data Generation")
print("=" * 60)
print(f"Project root: {project_root}")

## Step 1: Configuration

Configure the data generation parameters. You can either:
- Load from a JSON config file
- Set parameters directly in the notebook

Sampling method: `latin` (LHS), `random`, or `full_grid` / `structured_grid` / `grid`. Use `parameter_ranges` for grid; use `random_sample_bounds` for random/LHS.

In [ ]:
# Run control flags (metadata/data only)
IF_SAVE_METADATA = True  # Save metadata JSON when generating dataset
IF_SAVE_TRAINING_DATA = True  # Save training data (partial + final .pkl) to output_dir
IF_SAVE_COMPLETE_CSV = True   # Also write training_data_complete_*.csv (large); optional

# Load configuration from JSON (edit configs/ml/main2_data_generation_config.json)
CONFIG_FILE = 'configs/ml/main2_data_generation_config.json'

if not Path(CONFIG_FILE).exists():
    raise FileNotFoundError(f"Config not found: {CONFIG_FILE}")

with open(CONFIG_FILE, 'r') as f:
    config = json.load(f)
print(f"[OK] Loaded configuration from: {CONFIG_FILE}")

REACTANTS = config.get('reactants', ['ethane'])
MAX_COMBINATIONS = config.get('max_combinations_per_reactant', 100)
OUTPUT_DIR = config.get('output_dir', 'data/training')
SAVE_INTERVAL = config.get('save_interval', 10)
PARAM_RANGES = config.get('parameter_ranges', {})
SAMPLING_METHOD = config.get('sampling_method', 'latin_hypercube')
LHS_SEED = config.get('lhs_seed', 42)
RANDOM_SAMPLE_BOUNDS = config.get('random_sample_bounds', None)

print(f"  - Reactants: {REACTANTS}")
print(f"  - Max combinations per reactant: {MAX_COMBINATIONS}")
print(f"  - Output directory: {OUTPUT_DIR}")
print(f"  - Sampling method: {SAMPLING_METHOD}")

print(f"\nParameter Ranges:")
for param, values in PARAM_RANGES.items():
    if isinstance(values, list) and len(values) == 3:
        print(f"  - {param}: {values[0]} to {values[1]} ({values[2]} points)")
    else:
        print(f"  - {param}: {values}")

> 🧠 **Concept — Design of experiments: grid vs Latin Hypercube.** When every sample costs a full simulation, *where* you place your samples is a first-class design decision. A full grid with k points per parameter needs k^d runs: with the d = 6 parameters here, even a coarse 10-point grid means 10⁶ simulations, and the budget grows exponentially with every parameter you add. Worse, a grid spends that budget inefficiently — projected onto any single parameter, a million-run grid still visits only 10 distinct values. Latin Hypercube Sampling (LHS) inverts the deal: you fix the budget N first; the method slices each parameter's range into N equal strata, places exactly one sample in each, then shuffles the pairings between dimensions so the points also spread through the joint space. Every run then carries fresh information in *every* dimension at once — N runs give N distinct values of each parameter. Plain random sampling shares the fixed-budget property but leaves clumps and holes; LHS's stratification guarantees uniform 1D coverage. (Lesson 3 visualises the coverage this produces.)
>
> *In your domain:* the same trade appears in CFD parameter sweeps, perturbed-physics climate ensembles, and even ML hyperparameter search — random/LHS beats grid there for exactly this reason.

## Step 2: Initialize Data Generator

Create the data generator with your configuration.

In [ ]:
# Initialize generator
generator = TrainingDataGenerator(output_dir=OUTPUT_DIR, disable_plots=True)

# Update parameter ranges from config
if PARAM_RANGES:
    for key, value in PARAM_RANGES.items():
        if isinstance(value, list) and len(value) == 3:
            # Convert [min, max, n_points] to numpy array
            generator.param_ranges[key] = np.linspace(value[0], value[1], value[2])
            print(f"[OK] Updated {key}: {len(generator.param_ranges[key])} points")

# Calculate total combinations
total_combinations = generator._calculate_total_combinations()
print(f"\n[OK] Data generator initialized")
print(f"  - Output directory: {generator.output_dir}")
print(f"  - Total possible combinations: {total_combinations:,}")
print(f"  - Will generate: {len(REACTANTS) * MAX_COMBINATIONS:,} simulations")

#### 💪 Exercise 2.1 — grid vs LHS coverage, side by side

The cell above contrasts the total possible grid combinations (10 points per parameter over 6 parameters → 10⁶ per reactant) with the handful of runs this demo actually performs. The concept box above argued that LHS wins on coverage-per-run; check that claim yourself. Draw a small Latin Hypercube sample and a comparably-sized grid over the same two parameters, plot both, and compare how many distinct values of each parameter each design visits.

In [ ]:
# 💪 Exercise 2.1 — your turn (edit and re-run)
# TODO: change N_LHS_SAMPLES and GRID_POINTS_PER_AXIS to explore other budgets
import itertools

N_LHS_SAMPLES = 25
GRID_POINTS_PER_AXIS = 3   # a GRID_POINTS_PER_AXIS x GRID_POINTS_PER_AXIS grid, for a 2-parameter view

bounds_copy = {k: list(v) for k, v in RANDOM_SAMPLE_BOUNDS.items() if k != '_comment'}  # never edit RANDOM_SAMPLE_BOUNDS
axis_a, axis_b = 'temperature_K', 'heat_flux_Wm2'

# LHS draw, reusing the generator's own sampler (read-only call: does not touch `generator`'s state)
lhs_samples = generator.generate_parameter_combinations_lhs(
    n_samples=N_LHS_SAMPLES, random_sample_bounds=bounds_copy, seed=LHS_SEED,
)
lhs_a = [s[axis_a] for s in lhs_samples]
lhs_b = [s[axis_b] for s in lhs_samples]

# A small grid over the same two axes
grid_a = np.linspace(*bounds_copy[axis_a], GRID_POINTS_PER_AXIS)
grid_b = np.linspace(*bounds_copy[axis_b], GRID_POINTS_PER_AXIS)
grid_points = list(itertools.product(grid_a, grid_b))
grid_a_pts, grid_b_pts = zip(*grid_points)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(lhs_a, lhs_b, color='b')
axes[0].set_title(f'LHS, N={N_LHS_SAMPLES}')
axes[0].set_xlabel(axis_a); axes[0].set_ylabel(axis_b)
axes[1].scatter(grid_a_pts, grid_b_pts, color='r')
axes[1].set_title(f'Grid, {GRID_POINTS_PER_AXIS}x{GRID_POINTS_PER_AXIS} = {len(grid_points)} points')
axes[1].set_xlabel(axis_a); axes[1].set_ylabel(axis_b)
plt.tight_layout()
plt.show()

print(f"LHS: {len(set(lhs_a))} distinct {axis_a} values out of {N_LHS_SAMPLES} samples")
print(f"Grid: {len(set(grid_a_pts))} distinct {axis_a} values out of {len(grid_points)} samples")

<details><summary><b>💡 Solution & discussion — Exercise 2.1</b></summary>

```python
import itertools

N_LHS_SAMPLES = 25
GRID_POINTS_PER_AXIS = 3
bounds_copy = {k: list(v) for k, v in RANDOM_SAMPLE_BOUNDS.items() if k != '_comment'}
axis_a, axis_b = 'temperature_K', 'heat_flux_Wm2'

lhs_samples = generator.generate_parameter_combinations_lhs(
    n_samples=N_LHS_SAMPLES, random_sample_bounds=bounds_copy, seed=LHS_SEED,
)
lhs_a = [s[axis_a] for s in lhs_samples]
lhs_b = [s[axis_b] for s in lhs_samples]

grid_a = np.linspace(*bounds_copy[axis_a], GRID_POINTS_PER_AXIS)
grid_b = np.linspace(*bounds_copy[axis_b], GRID_POINTS_PER_AXIS)
grid_points = list(itertools.product(grid_a, grid_b))
```

The LHS scatter fills the rectangle with `N_LHS_SAMPLES` distinct values along *both* axes; the grid's points sit on a rigid `GRID_POINTS_PER_AXIS × GRID_POINTS_PER_AXIS` lattice — visually tidy, but only `GRID_POINTS_PER_AXIS` distinct values per parameter no matter how many total points you plot. Extend either design to all 6 parameters (this campaign's real dimensionality) and the gap widens: a grid with only 3 points per axis already needs 3⁶ = 729 runs for the same shallow per-parameter resolution that an LHS design of a few dozen runs achieves. This is why `main2_data_generation_config.json` ships with `"sampling_method": "latin"`.
</details>

### Step 2.1: Sampling preview moved to Main_3

EDA and training-space coverage plotting are handled in `Main_3_data_exploration_feature_engineering.ipynb`.

Preview how the parameter space will be explored: 1D marginals (uniformity) and 2D pairs (space filling).

In [ ]:
print("[INFO] Sampling preview removed from Main_2 for cleaner generation workflow.")
print("       Use Main_3_data_exploration_feature_engineering.ipynb for all EDA/coverage plots.")

## Step 3: Generate Training Data

Run the data generation process. This will:
- Run multiple PFR simulations with varied parameters
- Collect features and targets at each simulation point
- Save data incrementally to prevent loss
- Generate metadata for reproducibility

Note: This may take 5-30 minutes depending on the number of simulations.

> 🧠 **Concept — Embarrassingly parallel campaigns.** No simulation in this sweep depends on any other, which makes the campaign *embarrassingly parallel*: split the sampled conditions into chunks, run the chunks on whatever hardware is available — local cores, cluster nodes, cloud instances — and concatenate the results. This is precisely the SLURM job-array pattern on HPC systems, and it is how the course dataset was produced: 32 independent array tasks, each writing its own `training_data_complete_*.pkl`, consolidated afterwards into a single file with a merged metadata record. Two habits make campaigns like this safe at scale. Checkpointing: `save_interval` writes partial results every few runs, so a crash or a wall-time kill loses minutes of work, not hours. Metadata: the sampling method, seed, bounds and per-task provenance are written next to the data, so the campaign can be audited, reproduced or extended months later without archaeology.

In [ ]:
# Generate dataset
print("=" * 60)
print("Starting Training Data Generation...")
print("=" * 60)
print(f"Reactants: {REACTANTS}")
print(f"Max combinations per reactant: {MAX_COMBINATIONS}")
print(f"Sampling method: {SAMPLING_METHOD}")
print(f"Save interval: {SAVE_INTERVAL} simulations")
print("=" * 60)

start_time = time.time()

dataset = generator.generate_dataset(
    reactants=REACTANTS,
    max_combinations_per_reactant=MAX_COMBINATIONS,
    save_interval=SAVE_INTERVAL,
    random_sample_bounds=RANDOM_SAMPLE_BOUNDS,
    sampling_method=SAMPLING_METHOD,
    lhs_seed=LHS_SEED,
    save_metadata=IF_SAVE_METADATA,
    save_training_data=IF_SAVE_TRAINING_DATA,
    save_complete_csv=IF_SAVE_COMPLETE_CSV,
)

elapsed_time = time.time() - start_time
hours = int(elapsed_time // 3600)
minutes = int((elapsed_time % 3600) // 60)
seconds = int(elapsed_time % 60)

print(f"\n[OK] Data generation completed!")
print(f"  - Total time: {hours:02d}:{minutes:02d}:{seconds:02d}")

# Calculate actual number of simulations completed
# The dataset has multiple rows per simulation (one per reactor step)
# We get the actual count from the metadata file
if dataset is not None and len(dataset) > 0:
    import glob
    metadata_files = sorted(glob.glob(str(generator.output_dir / 'metadata_*.json')), reverse=True)
    if metadata_files:
        with open(metadata_files[0], 'r') as f:
            metadata = json.load(f)
        n_simulations = metadata.get('total_simulations', len(REACTANTS) * MAX_COMBINATIONS)
    else:
        n_simulations = len(REACTANTS) * MAX_COMBINATIONS
    
    print(f"  - Simulations completed: {n_simulations}")
    print(f"  - Data points collected: {len(dataset):,}")
    if n_simulations > 0:
        print(f"  - Average time per simulation: {elapsed_time / n_simulations:.2f} seconds")
        print(f"  - Data points per simulation: {len(dataset) / n_simulations:.1f}")
    else:
        print("  - Average time: N/A (no simulations completed)")
else:
    print("  - Average time: N/A (no data collected)")


> 🧠 **Concept — Failed runs are a fact of life.** In any large simulation campaign some fraction of runs will not finish: the stiff solver stalls at its minimum step size, a parameter combination turns out to be physically extreme, convergence tolerances are never met. The campaign behind this course attempted 50,000 simulations and completed 45,932 — a 91.9% success rate, which is entirely typical. The generator's job is to fail *gracefully*: record the failure together with the inputs that caused it, skip, and continue — run 4,317 must never be able to kill a 20-hour campaign. For ML, the bookkeeping matters more than the plumbing: failures are rarely uniform across the input space, and every failure is a hole in the training coverage. Keeping the failure log next to the dataset (the metadata JSON here does) is what lets you check, later, whether your surrogate's usable domain has quietly shrunk.
>
> *In your domain:* diverged CFD cases, crashed climate-ensemble members, non-converged DFT relaxations — same phenomenon, same rule: ship the failure log with the dataset.

#### 💪 Exercise 2.2 — read your own failure log

The generation run above wrote a metadata JSON file next to the dataset, whether or not any runs failed. Load it and check the actual success rate for *this* demo campaign — don't take the "10/10" for granted; inspect the file the way you would for a 50,000-run campaign where the failures matter.

In [ ]:
# 💪 Exercise 2.2 — your turn (edit and re-run)
# TODO: nothing required to run this — read the printed summary, then see the
#       discussion below for how to provoke real failures on a copy of the config
import glob

metadata_files = sorted(glob.glob(str(generator.output_dir / 'metadata_*.json')))
assert metadata_files, "No metadata file found — run the 'Generate Training Data' cell above first."

with open(metadata_files[-1]) as f:
    run_metadata = json.load(f)

print(f"Metadata file: {metadata_files[-1]}")
print(f"Total simulations attempted: {run_metadata['total_simulations']}")
print(f"Successful: {run_metadata['successful_simulations']}  |  "
      f"Failed: {run_metadata['failed_simulations']}  |  "
      f"Success rate: {run_metadata['success_rate']:.1f}%")

if run_metadata['failed_simulations'] > 0:
    print("\nThis demo run had failures — scroll back through the generation log above "
          "to see which parameter combinations were skipped, and why.")
else:
    print("\nNo failures in this small demo: every sampled point stayed inside a physically "
          "reasonable operating envelope. At the full campaign's 50,000-run scale (and wider "
          "bounds), a non-zero failure rate is normal — that's exactly what this metadata "
          "file is for.")

<details><summary><b>💡 Solution & discussion — Exercise 2.2</b></summary>

```python
import glob
metadata_files = sorted(glob.glob(str(generator.output_dir / 'metadata_*.json')))
with open(metadata_files[-1]) as f:
    run_metadata = json.load(f)
print(f"Successful: {run_metadata['successful_simulations']}/{run_metadata['total_simulations']} "
      f"({run_metadata['success_rate']:.1f}%)")
```

At this demo's scale, every run tends to succeed — the bounds in `random_sample_bounds` (`main2_data_generation_config.json`) were chosen to stay inside this mechanism's stable operating envelope. Push a bound outside that envelope — try widening `heat_flux_Wm2` well upward in a copy of the bounds, as in Exercise (2.3) — and you'll start seeing `[SKIP]` lines in the generation log above and a lower `success_rate` in this same file: the exact mechanism the concept box above describes at full campaign scale, just visible here in miniature.
</details>

#### 🤔 Think it through — nine million rows, but how many data points?

The output above reports ~200 data points per simulation; at full scale the campaign produced over 9 million rows from ~46,000 runs. When this data is split into training and test sets from Lesson 3 onwards, would a random 80/20 split over rows be acceptable? What is the effective sample size of this dataset?

<details><summary><b>💡 Discussion</b></summary>

No. The 200 rows of one run are near-siblings — identical inlet conditions, state varying smoothly along z. A random row-level split puts roughly 160 rows of each run in training and 40 in test, so the model is evaluated on close copies of examples it trained on, and the score becomes a flattering illusion (leakage). The unit of statistical independence is the run, not the row: the effective sample size is closer to 46,000 than to 9.2 million. Every split later in the course is therefore made at *run* level — a run's rows all go to train or all to test. This is the single most common trap when ML practitioners first meet simulation, time-series or patient-level data, and it should change how you read every metric in the rest of the course.
</details>

#### 💪 Exercise (2.3) — run your own tiny campaign

Everything above ran the demo campaign fixed by `configs/ml/main2_data_generation_config.json`. Copy the bounds, push them somewhere new, and run a genuinely new — but tiny — campaign of your own: 3 simulations, written to a scratch folder that can never collide with `data/training/` or with the course dataset.

In [ ]:
# 💪 Exercise (2.3) — your turn (edit and re-run)
# TODO: change SCRATCH_BOUNDS to explore a different corner of the parameter space
import copy

SCRATCH_BOUNDS = copy.deepcopy(RANDOM_SAMPLE_BOUNDS)   # copy: never edit RANDOM_SAMPLE_BOUNDS itself
SCRATCH_BOUNDS.pop('_comment', None)
SCRATCH_BOUNDS['temperature_K'] = [880, 900]            # push toward the hot end of the range

scratch_generator = TrainingDataGenerator(
    output_dir=str(project_root / 'outputs' / 'scratch' / 'main2_exercise'),  # never data/training/
    disable_plots=True,
)

scratch_dataset = scratch_generator.generate_dataset(
    reactants=REACTANTS,
    max_combinations_per_reactant=3,     # tiny — just enough to see the mechanics
    save_interval=3,
    random_sample_bounds=SCRATCH_BOUNDS,
    sampling_method='latin',
    lhs_seed=LHS_SEED,
    save_metadata=True,
    save_training_data=True,
    save_complete_csv=False,
)

n_rows = len(scratch_dataset) if scratch_dataset is not None else 0
print(f"\nScratch campaign: {n_rows} rows written under {scratch_generator.output_dir} "
      f"(separate from data/training/)")

<details><summary><b>💡 Solution & discussion — Exercise (2.3)</b></summary>

```python
import copy

SCRATCH_BOUNDS = copy.deepcopy(RANDOM_SAMPLE_BOUNDS)
SCRATCH_BOUNDS.pop('_comment', None)
SCRATCH_BOUNDS['temperature_K'] = [880, 900]

scratch_generator = TrainingDataGenerator(
    output_dir=str(project_root / 'outputs' / 'scratch' / 'main2_exercise'),
    disable_plots=True,
)
scratch_dataset = scratch_generator.generate_dataset(
    reactants=REACTANTS,
    max_combinations_per_reactant=3,
    save_interval=3,
    random_sample_bounds=SCRATCH_BOUNDS,
    sampling_method='latin',
    lhs_seed=LHS_SEED,
)
```

Three runs are obviously too few to train anything on, but they're enough to see the whole campaign mechanism working end to end: parameter sampling, per-run Cantera simulation, incremental saves, and a metadata file — all landing under `outputs/scratch/main2_exercise/`, well clear of `data/training/`, so it can't collide with (or be mistaken for) the course dataset. Scaling this up from here is purely a matter of raising `max_combinations_per_reactant`; at real campaign scale, the concept box above describes how the same `all_tasks` list gets split across SLURM array workers instead of running sequentially.
</details>